# Jobs: fit/fixed sigma dt

host = ```yoru/chewie```

**Motivation**: <br>


In [1]:
# HIDE CODE


project_name = '_TemporalSC'


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, project_name))
from figures.convergence import plot_convergence
from main.config_defaults import default_configs
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = pjoin(git_dir, project_name, 'scripts')
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'cleanup_chkpts.sh',
    'cleanup_recursive.sh',
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## yoru

recon_mode = `mse` (default)

In [4]:
host = 'yoru'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
t_train = 16
recon_mode = 'mse'

betas = [
    0.5, 1, 2, 4, 8,
    12, 16, 24, 32,
]
fit_dt_sigma = list(itertools.product(
    (True, False), repeat=2))

len(fit_dt_sigma), len(betas)

(4, 9)

In [6]:
for kl_beta in betas:
    for fit_dt, fit_sigma in fit_dt_sigma:
        # get arg
        arg = ' '.join([
            f"--t_train {t_train}",
            f"--kl_beta {kl_beta}",
            f"--fit_dt {str(fit_dt).lower()}",
            f"--fit_sigma {str(fit_sigma).lower()}",
            f"--recon_mode '{recon_mode}'",
            f"--comment 'recon-{recon_mode}_dt-{fit_dt}-sigma-{fit_sigma}'",
            '--verbose',
            # '--dry_run',
        ])
        gpu_i = tot % torch.cuda.device_count()
        scripts[gpu_i].append(job_runner_script(
            device=gpu_i,
            dataset='vH16-wht',
            model='poisson',
            archi='ngd|lin',
            args=arg,
            seed=0,
        ))
        tot += 1

In [7]:
print(tot)

36

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 18, 1: 18}

### Save

In [9]:
n_fits = 3

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'yoru-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt true --fit_sigma 
true --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt false --fit_sigma 
true --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt true --fit_sigma true 
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt false --fit_sigma true
--recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt true --fit_sigma true 
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt false --fit_sigma true
--recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-True' --verbose

[PROGRESS] 'yoru-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt true --fit_sigma true 
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt false --fit_sigma true
--recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt true --fit_sigma true 
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt false --fit_sigma true
--recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt true --fit_sigma true
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt false --fit_sigma 
true --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-True' --verbose

[PROGRESS] 'yoru-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

...

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt true --fit_sigma false
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt true --fit_sigma false
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose

[PROGRESS] 'yoru-cuda1-fit1.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt true --fit_sigma false
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt true --fit_sigma false
--recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose

[PROGRESS] 'yoru-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt true --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt false --fit_sigma 
false --recon_mode 'mse' --comment 'recon-mse_dt-False-sigma-False' --verbose

## chewie

recon_mode = `prob`

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
t_train = 16
recon_mode = 'prob'

betas = [
    0.5, 1, 2, 4, 8,
    12, 16, 24, 32,
]
fit_dt_sigma = list(itertools.product(
    (True, False), repeat=2))

len(fit_dt_sigma), len(betas)

(4, 9)

In [6]:
for kl_beta in betas:
    for fit_dt, fit_sigma in fit_dt_sigma:
        # get arg
        arg = ' '.join([
            f"--t_train {t_train}",
            f"--kl_beta {kl_beta}",
            f"--fit_dt {str(fit_dt).lower()}",
            f"--fit_sigma {str(fit_sigma).lower()}",
            f"--recon_mode '{recon_mode}'",
            f"--comment 'recon-{recon_mode}_dt-{fit_dt}-sigma-{fit_sigma}'",
            '--verbose',
            # '--dry_run',
        ])
        gpu_i = tot % torch.cuda.device_count()
        scripts[gpu_i].append(job_runner_script(
            device=gpu_i,
            dataset='vH16-wht',
            model='poisson',
            archi='ngd|lin',
            args=arg,
            seed=0,
        ))
        tot += 1

In [7]:
print(tot)

36

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 18, 1: 18}

### Save

In [9]:
n_fits = 3

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'chewie-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt true --fit_sigma 
true --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt false --fit_sigma 
true --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt true --fit_sigma true 
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt false --fit_sigma true
--recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt true --fit_sigma true 
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt false --fit_sigma true
--recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-True' --verbose

[PROGRESS] 'chewie-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt true --fit_sigma true 
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt false --fit_sigma true
--recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt true --fit_sigma true 
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt false --fit_sigma true
--recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt true --fit_sigma true
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-True' --verbose && 
./fit_model.sh '0' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt false --fit_sigma 
true --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-True' --verbose

[PROGRESS] 'chewie-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

...

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 0.5 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt true --fit_sigma false
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 1 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt true --fit_sigma false
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 2 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose

[PROGRESS] 'chewie-cuda1-fit1.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt true --fit_sigma false
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 4 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt true --fit_sigma false
--recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 8 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 12 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose

[PROGRESS] 'chewie-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/_TemporalSC/scripts

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 16 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 24 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt true --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-True-sigma-False' --verbose && 
./fit_model.sh '1' 'vH16-wht' 'poisson' 'ngd|lin' --seed 0 --t_train 16 --kl_beta 32 --fit_dt false --fit_sigma 
false --recon_mode 'prob' --comment 'recon-prob_dt-False-sigma-False' --verbose